# Nuclear Pore Complex iMSM Tutorial (Flagship Pipeline)

This notebook demonstrates the end-to-end **interaction Markov State Model (iMSM)** pipeline for analyzing nucleocytoplasmic transport trajectories through the Nuclear Pore Complex (NPC).

### Physical Representation & Method Overview
iMSMs automate the identification of recurring interaction states and the estimation of their transition network within a user-specified physical representation:
1. **Partitioning the System**: The simulated NPC system is partitioned into a set of discrete interacting components $\mathcal{C} = \{c_1, \ldots, c_N\}$ (the FG-nucleoporin beads across the 8-fold symmetric pore scaffold) and a focal entity $f$ (a translocating cargo:NTR or Karyopherin complex) that interacts with subsets of $\mathcal{C}$.
2. **Interaction Trajectory $I(t)$**: Each Brownian dynamics trajectory is converted into an interaction trajectory $I(t)$ recording contacts between $f$ and $\mathcal{C}$ within a spatial contact cutoff (`MAX_SURFACE_DIST_NM` / `DISTANCE_STATE_THRESHOLD_NM`). At each frame, up to a reference `INTERACTION_CAPACITY` closest contacts are retained.
3. **Windowed Interaction Histograms $H$**: $I(t)$ is divided into consecutive time windows of duration $\tau$ (`WINDOW_SIZE_STEPS`). The interaction statistics in each window are summarized in an interaction histogram $H$, normalized by the fixed reference interaction capacity for the focal entity (rather than observed contacts). This distinguishes strongly, partially, and weakly engaged states even when involving similar partners.
4. **Unsupervised Symmetric Clustering**: Interaction histograms are clustered into a reduced set of recurring interaction states $\mathcal{S} = \{s_1, \ldots, s_k\}$ using an 8-fold rotational symmetry-aware algorithm (`sym-faiss`), accommodating the C8 architectural symmetry of the NPC.
5. **Lagged Transition Matrix & Transport Kinetics**: Transitions between states at lag time $\tau$ are recorded to estimate the transition-probability matrix:
   $$T_{ij} = P(s(t+\tau) = s_j \mid s(t) = s_i)$$
   From $T_{ij}$, stationary distributions, axial free-energy barrier profiles, nucleoplasmic/cytoplasmic committor probabilities, and effective pore permeabilities are computed.

**Input Data Reference:**
B. Raveh, R. Eliasian, S. Rashkovits, D. Russel, R. Hayama, S. Sparks, D. Singh, R.Y.H. Lim, E. Villa, M.P. Rout, D. Cowburn, & A. Sali, *Integrative mapping reveals molecular features underlying the mechanism of nucleocytoplasmic transport*, Proc. Natl. Acad. Sci. U.S.A. 122 (42) e2507559122 (2025). https://doi.org/10.1073/pnas.2507559122


## 1. Import Required Libraries

In [ ]:
from iMSM.extensions.npc.npc import run

## 2. Input Data Format & Pipeline Stages

### Input Data Format
If using the bundled `.rmf` trajectory loaders, arrange input simulation files as follows:
- A parent directory containing numbered subdirectories for each independent simulation run (e.g. `simulations/1/`, ..., `simulations/30/`), specified via `LOAD_MD_SIMS_RANGE`.
- Each folder contains `.rmf` timeframes spaced by `LOAD_MD_STEP_NS`, bounded by `LOAD_MD_START_TIME_NS` and `LOAD_MD_END_TIME_NS`.
- Focal transporter particles are named `kap<radius>` (matching `LOAD_MD_KAP_RADIUS`). FG-nucleoporin bead types belong to `["Nup2", "Nsp1", "Nup100", "Nup116", "Nup159", "Nup49", "Nup57", "Nup145", "Nup1", "Nup60"]` (unmodeled types listed in `LOAD_MD_IGNORED_NUP_TYPES`).

### Five-Step iMSM Pipeline Stages
Execution proceeds through stages 1 to 7 via `run(..., start_stage=1, end_stage=8)`:
1. **Stage 1 (`stage_01_loadNTRs`) — Extract Focal Entity Coordinates**: Extracts 3D coordinates of focal Kap particles from `.rmf` files into per-simulation caches.
2. **Stage 2 (`stage_02_loadFGs`) — Extract Interacting Component Coordinates**: Extracts 3D coordinates for all FG-nucleoporin beads across trajectories.
3. **Stage 3 (`stage_03_categorize`) — Construct Interaction Trajectory $I(t)$**: Evaluates pairwise distances between focal Kap $f$ and FG beads in $\mathcal{C}$ against the contact threshold `MAX_SURFACE_DIST_NM`, recording up to `INTERACTION_CAPACITY` closest contacts per frame.
4. **Stage 4 (`stage_04_embed`) — Windowed Histogram Embedding $H$**: Aggregates $I(t)$ over consecutive time windows of duration $\tau$ (`WINDOW_SIZE_STEPS`) into interaction histograms normalized by reference `INTERACTION_CAPACITY`.
5. **Stage 5 (`stage_05_cluster`) — Unsupervised 8-Fold Symmetric Clustering**: Clusters histograms into recurring interaction mesostates $\mathcal{S}$ using `sym-faiss`.
6. **Stage 6 (`stage_06_buildMSM`) — Estimate Transition-Probability Matrix $T_{ij}$**: Infers the lagged transition matrix $T_{ij}$, stationary distribution, and flux pathways.
7. **Stage 7 (`stage_07_computePermeabilities`) — Transport Observables**: Calculates macroscopic transport observables: transit times, exit committor probabilities, and effective pore permeability.


## 3. Configure and Run iMSM

In [ ]:
params = {}
checkpoint_path = ...

# --- System & Trajectory Parameters ---
params['LOAD_MD_BASE_PATH'] = ...
params['LOAD_MD_KAP_SITES'] = ...       # Number of interaction sites on the focal entity (Kap)
params['LOAD_MD_KAP_RADIUS'] = ...      # Radius of the focal entity (Kap) in Angstroms
params['LOAD_MD_KAP_AMOUNT'] = 100      # Number of focal entities per simulation
params['LOAD_MD_SIMS_RANGE'] = range(1, 31)  # Range of independent simulation trajectories
params['LOAD_MD_START_TIME_NS'] = 10000 # Simulation start time in ns (e.g. 10 us)
params['LOAD_MD_END_TIME_NS'] = 30000   # Simulation end time in ns (e.g. 30 us)
params['LOAD_MD_STEP_NS'] = 100         # Time step between trajectory frames in ns

# --- iMSM Representation & Discretization Parameters ---
params['DISTANCE_STATE_THRESHOLD_NM'] = 10 # Contact definition cutoff distance between f and components in C
params['INTERACTION_CAPACITY'] = 5         # Fixed reference interaction capacity for normalizing histograms H
params['WINDOW_SIZE_STEPS'] = 10           # Duration tau of consecutive time windows (frames)

# --- Clustering & MSM Estimation Parameters ---
params['N_CLUSTERS'] = [320]               # Target number of recurring interaction states (mesostates)
params['TM_PRIOR'] = 0.01                  # Transition matrix pseudo-count prior

# --- Data Subsetting & Transport Observables ---
params['MODE'] = "subset"
params['DATA_SUBSET'] = 1.00
params['DATA_SUBSET_INDEX'] = 0
params['DATA_SUBSET_MODE'] = "simulation"
params['STATE_CHOICE_METHOD'] = "distance" # Distance-based boundary definition for permeability calculation

# Execute stages 1 to 7
run(checkpoint_path, params_override=params, start_stage=1, end_stage=8)
